In [1]:
import sys
import os
project_root = os.path.abspath('..')
sys.path.append(project_root)
from src import AdversarialObjectDetection, add_noise, compare_detections, test_noise_defense_with_iou
import torch
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
detector = AdversarialObjectDetection()
image_url = "..\\leftImg8bit\\test\\berlin\\berlin_000000_000019_leftImg8bit.png"
TARGET_CLASS = 20
noise_configs = [
    {'type': 'gaussian', 'mean': 0, 'std': 5},
    {'type': 'gaussian', 'mean': 0, 'std': 10},
    {'type': 'gaussian', 'mean': 0, 'std': 15},
    {'type': 'salt_and_pepper', 'density': 0.05},
    {'type': 'salt_and_pepper', 'density': 0.1},
    {'type': 'speckle', 'intensity': 0.2},
    {'type': 'poisson', 'scale': 1.0}
]
results = test_noise_defense_with_iou(
    detector=detector,
    image_path=image_url,
    target_class=TARGET_CLASS,
    noise_configs=noise_configs,
    num_iterations=3
)

c:\Users\victo\Documents\Workspaces\thesis_project\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\victo\Documents\Workspaces\thesis_project\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model
Iteration 0: Loss = 14.8235


In [3]:
print(results['noise_tests'][0]['adversarial_comparison']['average_iou'])

0.6321835


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
detector = AdversarialObjectDetection()
import glob
test_images = glob.glob("..\\leftImg8bit\\test\\berlin\\*.png")[:100]  
print(f"Found {len(test_images)} test images")

# Test parameters
TARGET_CLASS = 20
noise_configs = [
    {'type': 'gaussian', 'mean': 0, 'std': 5},
    {'type': 'gaussian', 'mean': 0, 'std': 10},
    {'type': 'gaussian', 'mean': 0, 'std': 15},
    {'type': 'salt_and_pepper', 'density': 0.05},
    {'type': 'salt_and_pepper', 'density': 0.1},
    {'type': 'speckle', 'intensity': 0.2},
    {'type': 'poisson', 'scale': 1.0}
]

iou_types = ["standard", "giou"]

Loading model
Found 100 test images


In [ ]:
def batch_test_with_loops(detector, image_paths, target_class, noise_configs, 
                          iou_types, num_iterations=3):
    """
    Loop through images and IoU types, collect all IoU scores
    Fixed IoU threshold just for matching, not for filtering analysis
    """
    all_results = []
    total_tests = len(image_paths) * len(iou_types)
    test_count = 0
    
    for image_path in image_paths:
        image_name = Path(image_path).name
        
        for iou_type in iou_types:
            test_count += 1
            print(f"[{test_count}/{total_tests}] IoU Type: {iou_type}")
            
            try:
                result = test_noise_defense_with_iou(
                    detector=detector,
                    image_path=image_path,
                    target_class=target_class,
                    noise_configs=noise_configs,
                    num_iterations=num_iterations,
                    iou_type=iou_type
                )
                
                result['image'] = image_name
                result['iou_type'] = iou_type
                
                all_results.append(result)
                
            except Exception as e:
                print(f"  ERROR: {e}")
                all_results.append({
                    'image': image_name,
                    'iou_type': iou_type,
                    'error': str(e)
                })
    
    return all_results



In [8]:
def compare_iou_types_plot(all_results):
    """
    Compare distributions across different IoU calculation methods
    """
    iou_type_data = {}
    
    for result in all_results:
        if 'error' in result:
            continue
        
        iou_type = result['iou_type']
        if iou_type not in iou_type_data:
            iou_type_data[iou_type] = {'clean': [], 'adversarial': []}
        
        for noise_test in result['noise_tests']:
            clean_comp = noise_test['clean_comparison']
            adv_comp = noise_test['adversarial_comparison']
            
            iou_type_data[iou_type]['clean'].extend(clean_comp.get('matched_iou_values', []))
            iou_type_data[iou_type]['adversarial'].extend(adv_comp.get('matched_iou_values', []))
    
    # Plot comparison
    num_types = len(iou_type_data)
    fig, axes = plt.subplots(num_types, 2, figsize=(16, 6*num_types))
    
    if num_types == 1:
        axes = axes.reshape(1, -1)
    
    bins = np.arange(0, 1.05, 0.05)
    
    for idx, (iou_type, values) in enumerate(iou_type_data.items()):
        # Clean
        axes[idx, 0].hist(values['clean'], bins=bins, alpha=0.7, color='blue', edgecolor='black')
        axes[idx, 0].set_xlabel('IoU Score')
        axes[idx, 0].set_ylabel('Frequency')
        axes[idx, 0].set_title(f'{iou_type.upper()} - Clean Images (n={len(values["clean"])}, mean={np.mean(values["clean"]):.3f})')
        axes[idx, 0].grid(True, alpha=0.3)
        axes[idx, 0].set_xlim(0, 1)
        
        # Adversarial
        axes[idx, 1].hist(values['adversarial'], bins=bins, alpha=0.7, color='red', edgecolor='black')
        axes[idx, 1].set_xlabel('IoU Score')
        axes[idx, 1].set_ylabel('Frequency')
        axes[idx, 1].set_title(f'{iou_type.upper()} - Adversarial Images (n={len(values["adversarial"])}, mean={np.mean(values["adversarial"]):.3f})')
        axes[idx, 1].grid(True, alpha=0.3)
        axes[idx, 1].set_xlim(0, 1)
    
    plt.tight_layout()
    plt.savefig('iou_type_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

all_results = batch_test_with_loops(
    detector=detector,
    image_paths=test_images,
    target_class=TARGET_CLASS,
    noise_configs=noise_configs,
    iou_types=iou_types,  
    num_iterations=3 
)

compare_iou_types_plot(all_results)

[1/200] IoU Type: standard
  ERROR: test_noise_defense_with_iou() got an unexpected keyword argument 'iou_threshold'
[2/200] IoU Type: giou
  ERROR: test_noise_defense_with_iou() got an unexpected keyword argument 'iou_threshold'
[3/200] IoU Type: standard
  ERROR: test_noise_defense_with_iou() got an unexpected keyword argument 'iou_threshold'
[4/200] IoU Type: giou
  ERROR: test_noise_defense_with_iou() got an unexpected keyword argument 'iou_threshold'
[5/200] IoU Type: standard
  ERROR: test_noise_defense_with_iou() got an unexpected keyword argument 'iou_threshold'
[6/200] IoU Type: giou
  ERROR: test_noise_defense_with_iou() got an unexpected keyword argument 'iou_threshold'
[7/200] IoU Type: standard
  ERROR: test_noise_defense_with_iou() got an unexpected keyword argument 'iou_threshold'
[8/200] IoU Type: giou
  ERROR: test_noise_defense_with_iou() got an unexpected keyword argument 'iou_threshold'
[9/200] IoU Type: standard
  ERROR: test_noise_defense_with_iou() got an unexpec

ValueError: Number of rows must be a positive integer, not 0

<Figure size 1600x0 with 0 Axes>